# 00 · Preparar el entorno

Antes de empezar comprobamos **qué tenemos** (versiones, núcleos de CPU) y **qué vamos a usar** (licencias), y descargamos una vez los pesos de los modelos a la caché (`HF_HOME`), para que los notebooks siguientes no dependan de la red.

| Pieza | Para qué | Licencia |
|---|---|---|
| SmolVLM-256M-Instruct | el LLM de visión (A y B) | se lee de su ficha de Hugging Face (abajo) |
| MobileNetV3-small (torchvision, ImageNet) | línea base C | código de torchvision BSD-3-Clause; los pesos: ver la documentación de torchvision |
| Oxford-IIIT Pet | las fotos | CC BY-SA 4.0 (Parkhi et al., CVPR 2012) |

Salida: `entorno.json` en la carpeta de trabajo.

In [1]:
import os
N_EJEMPLOS = int(os.environ.get("ESTILO_N_EJEMPLOS", "60"))
EPOCAS = int(os.environ.get("ESTILO_EPOCAS", "2"))
RESOLUCION = int(os.environ.get("ESTILO_RESOLUCION", "224"))
MODELO = os.environ.get("ESTILO_MODELO", "HuggingFaceTB/SmolVLM-256M-Instruct")
CARPETA_TRABAJO = os.environ.get("ESTILO_TRABAJO", "_trabajo")

In [2]:
import json, platform
from importlib.metadata import version
import comun_estilo as ce

trabajo = ce.carpeta_trabajo(CARPETA_TRABAJO)
hilos = ce.limitar_hilos()
entorno = {"python": platform.python_version(), "sistema": platform.system(), "nucleos_logicos": os.cpu_count(), "hilos_torch": hilos}
for paquete in ("torch", "torchvision", "transformers", "peft", "accelerate", "huggingface_hub", "mlflow",
                "scikit-learn", "scikit-image", "numpy", "pillow", "matplotlib"):
    entorno[paquete] = version(paquete)
for k, v in entorno.items():
    print(f"{k:16s} {v}")

python           3.11.18
sistema          Windows
nucleos_logicos  14
hilos_torch      12
torch            2.15.0+cpu
torchvision      0.29.2+cpu
transformers     5.23.0
peft             0.22.0
accelerate       1.16.0
huggingface_hub  2.4.0
mlflow           3.18.0
scikit-learn     1.9.1
scikit-image     0.26.0
numpy            2.4.6
pillow           13.0.0
matplotlib       3.11.2


## Licencia y descarga del VLM

Leemos la licencia de la **ficha del modelo** en Hugging Face (no la suponemos) y descargamos sus archivos. Si ya estaban en la caché, la «descarga» tarda poco.

In [3]:
from huggingface_hub import model_info, snapshot_download
try:
    info = model_info(MODELO)
    entorno["licencia_modelo"] = (info.card_data or {}).get("license", "no se pudo comprobar")
except Exception as e:
    entorno["licencia_modelo"] = f"no se pudo comprobar ({type(e).__name__})"
with ce.Cronometro() as c:
    carpeta_modelo = snapshot_download(MODELO, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja"])
entorno["s_descarga_modelo"] = round(c.s, 1)
entorno["mb_modelo_en_cache"] = ce.tamano_mb(carpeta_modelo)
print(f"{MODELO}: licencia «{entorno['licencia_modelo']}»; {entorno['mb_modelo_en_cache']} MB; {c.s:.1f} s")

from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
with ce.Cronometro() as c:
    mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
entorno["s_descarga_mobilenet"] = round(c.s, 1)
print(f"MobileNetV3-small (ImageNet): {c.s:.1f} s")
(trabajo / "entorno.json").write_text(json.dumps(entorno, indent=1), encoding="utf-8")

HuggingFaceTB/SmolVLM-256M-Instruct: licencia «apache-2.0»; 493.9 MB; 0.1 s


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to C:\bk\lab-estilo\hf\torch\hub\checkpoints\mobilenet_v3_small-047dcff4.pth


  0%|          | 0.00/9.83M [00:00<?, ?B/s]

 10%|█         | 1.00M/9.83M [00:00<00:00, 9.79MB/s]

 20%|██        | 2.00M/9.83M [00:00<00:00, 9.59MB/s]

 31%|███       | 3.00M/9.83M [00:00<00:00, 8.87MB/s]

... (5 outputs left out)

 99%|█████████▉| 9.75M/9.83M [00:01<00:00, 10.6MB/s]

100%|██████████| 9.83M/9.83M [00:01<00:00, 10.2MB/s]

MobileNetV3-small (ImageNet): 1.3 s


505